# FlexiStaff · team-aqua : entraînement YOLO tracé dans MLflow

Notebook Colab du jour 1. Il couvre pour l'instant les étapes du guide étudiant jusqu'à **5.4.2** :

1. **5.4.1** préparer le notebook (GPU, dépendances) ;
2. **5.3** préparer le jeu de données à partir de l'export CVAT (découpage train / val) ;
3. **5.4.2** se connecter à MLflow et sélectionner l'expérience `flexistaff_team-aqua`.

L'entraînement (5.5) et l'enregistrement du champion (5.6) viendront à la suite.

> Le fichier `identifiants-<login>.json` contient vos mots de passe : ne le committez pas et ne le laissez pas dans un notebook partagé.

## 5.4.1 Préparer le notebook

Activez d'abord le GPU : **Exécution > Modifier le type d'exécution > GPU (T4)**. Changer de type d'exécution réinitialise la session : faites-le **avant** d'installer quoi que ce soit.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : vérifiez le type d'exécution"

In [ ]:
%pip install -q ultralytics mlflow onnx onnxruntime

## 5.3 Préparer le jeu de données

Le chef d'équipe exporte la tâche entière depuis CVAT au format **Ultralytics YOLO Detection 1.0** et partage l'archive `.zip`.
Déposez-la dans Colab (panneau **Fichiers**, à gauche), puis indiquez son nom ci-dessous.

Le script `ai_training/prepare_dataset.py` du dépôt redécoupe les images en train / val (80 / 20, tirage fixe) et écrit `/content/dataset/data.yaml` avec les noms de classes de l'export, dans l'ordre de CVAT.

In [ ]:
import glob, os, sys

REPO = "/content/flexistaff"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/TangiLC/flexistaff {REPO}
sys.path.insert(0, f"{REPO}/ai_training")

from prepare_dataset import prepare

EXPORT = None          # ex. "/content/task_lot-1.zip" ; None = premier .zip trouvé dans /content
VAL_RATIO = 0.2
SEED = 42

if EXPORT is None:
    zips = sorted(glob.glob("/content/*.zip"))
    assert zips, "Déposez l'archive exportée depuis CVAT dans /content"
    EXPORT = zips[0]
print("Export utilisé :", EXPORT)

DATA_YAML = prepare(EXPORT, "/content/dataset", VAL_RATIO, SEED)
print(open(DATA_YAML).read())

Vérifiez la sortie : le nombre de boîtes par classe en `train` et en `val`. Une classe absente du `val` ne pourra pas être évaluée ; si c'est le cas, changez `SEED` ou signalez-le à la squad.

## 5.4.2 Se connecter à MLflow

MLflow est protégé par le portail : le client doit présenter vos identifiants (les mêmes que pour CVAT). Les artefacts transitent par le serveur MLflow, pas besoin des identifiants S3.

**Méthode recommandée** : téléchargez `identifiants-<login>.json` depuis votre profil du portail et déposez-le dans Colab. Sans le fichier, la cellule demande le mot de passe au clavier (renseignez alors `LOGIN`).

In [ ]:
import glob, json, os
from getpass import getpass
import mlflow

SQUAD = "team-aqua"
EXPERIMENT = f"flexistaff_{SQUAD}"
LOGIN = None           # ex. "prenom.n" : seulement utile sans fichier d'identifiants

fichiers = sorted(glob.glob("/content/identifiants-*.json")) or sorted(glob.glob("identifiants-*.json"))
if fichiers:
    creds = json.load(open(fichiers[0]))["mlflow"]
    os.environ["MLFLOW_TRACKING_USERNAME"] = creds["username"]
    os.environ["MLFLOW_TRACKING_PASSWORD"] = creds["password"]
    mlflow.set_tracking_uri(creds["tracking_uri"])
    print("Identifiants lus dans", os.path.basename(fichiers[0]))
else:
    assert LOGIN, "Pas de fichier identifiants-*.json : renseignez LOGIN"
    os.environ["MLFLOW_TRACKING_USERNAME"] = LOGIN
    os.environ["MLFLOW_TRACKING_PASSWORD"] = getpass("Mot de passe : ")
    mlflow.set_tracking_uri("http://13.37.102.106:5000")

exp = mlflow.set_experiment(EXPERIMENT)
print(f"Connecté à {mlflow.get_tracking_uri()} · expérience {exp.name} (id {exp.experiment_id})")

In [ ]:
# Vérification : les runs déjà présents dans l'expérience de la squad (vide au premier passage)
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], max_results=20,
                          order_by=["attributes.start_time DESC"])
cols = [c for c in ["tags.mlflow.runName", "metrics.recall", "metrics.mAP50", "start_time"] if c in runs.columns]
runs[cols] if len(runs) else print("Aucun run pour l'instant.")

Conventions pour la suite (5.5) :

| Élément d'un run | Ce qu'on y met |
|---|---|
| `params` | modèle, epochs, taille d'image, version du dataset |
| `metrics` | mAP50, précision, rappel |
| `artifacts` | `best.pt`, courbes, modèle ONNX |
| `tags` | `squad`, `dataset` |

Nom du run : `<membre>_<version>` (ex. `simon_v1`). Tous les membres tracent dans `flexistaff_team-aqua`.